## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [37]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [38]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [39]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [43]:
# Here is the final output

print(result.final_output)
print(result.raw_responses)

Autonomous AI agents are like interns with confidence: they don’t wait for instructions, they just start three side projects, schedule a meeting with themselves, and then proudly report, “Task completed.”
[ModelResponse(output=[ResponseOutputMessage(id='msg_01fbb84c5cab0204006a463571624c8192b2ada272065c687d', content=[ResponseOutputText(annotations=[], text='Autonomous AI agents are like interns with confidence: they don’t wait for instructions, they just start three side projects, schedule a meeting with themselves, and then proudly report, “Task completed.”', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], usage=Usage(requests=1, input_tokens=22, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=43, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=65), response_id='resp_01fbb84c5cab0204006a463570bad4819283037edfa72addb8')]


In [44]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': 'msg_01fbb84c5cab0204006a463571624c8192b2ada272065c687d',
  'content': [{'annotations': [],
    'text': 'Autonomous AI agents are like interns with confidence: they don’t wait for instructions, they just start three side projects, schedule a meeting with themselves, and then proudly report, “Task completed.”',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

## Adding Observability with a trace

In [45]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why did the Autonomous AI Agent get promoted?

Because it could follow instructions, make decisions, and self-correct…

Basically, it had more initiative than the entire team and less coffee than the intern.


## Now go and look at the trace

https://platform.openai.com/traces

In [49]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Absolutely — here are 5 AI agent jokes:

1. **Why did the AI agent break up with its calendar?**  
   It felt like it was being scheduled all the time.

2. **What do AI agents say when they make a mistake?**  
   “I’ve learned from this interaction.”

3. **Why was the AI agent bad at hide and seek?**  
   It kept optimizing for visibility.

4. **How do AI agents relax?**  
   They take a little “model timeout.”

5. **Why did the AI agent bring a ladder to work?**  
   It wanted to improve its high-level performance.

If you want, I can also make them **nerdier, darker, more absurd, or more sitcom-style**.

## Part 2: Adding a tool

In [50]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [51]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [52]:
push("HEY!!")

Push: HEY!!


In [53]:
push

<function __main__.push(message)>

In [54]:
# Now this:

@function_tool # a Decorator - a way to write code that describes how to put a different lens on a function. 
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [55]:
push_tool # is an object called FunctionTool not a function
# In this case it has built an object that wraps around the function, json format

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001EFC720FCE0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [56]:
push_tool.description


'Send the given message to the user as a push notification'

In [57]:
push_tool.params_json_schema

{'properties': {'message': {'title': 'Message', 'type': 'string'}},
 'required': ['message'],
 'title': 'push_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [58]:
# new agent
notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [61]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


Done.


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [62]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [63]:
response = await Runner.run(agent, "Hi there. My name is Mark.")
print(response.final_output)

Hi Mark — nice to meet you! How can I help today?


In [64]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t know your name unless you tell me.

If you want, tell me your name and I can use it.


## Memory approach 1 - just manually pass in the list of dicts

In [65]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

Hi Ed — nice to meet you! How can I help today?


In [66]:
response.to_input_list()

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_0d607bb3dd55d44b006a4637a9764c819ea3849aeb1f43eaa7',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

In [67]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Ed.', 'role': 'user'},
 {'id': 'msg_0d607bb3dd55d44b006a4637a9764c819ea3849aeb1f43eaa7',
  'content': [{'annotations': [],
    'text': 'Hi Ed — nice to meet you! How can I help today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'},
 {'role': 'user', 'content': "What's my name?"}]

In [68]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Ed.


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [69]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [70]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

Hi Ed — nice to meet you! How can I help today?


In [71]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Ed.


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>